In [4]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "Clean_Dataset.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "shubhambathwal/flight-price-prediction",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

df = df.rename(columns={"Unnamed: 0": "ID"})
df["route"] = df["source_city"] + " -> " + df["destination_city"]

In [4]:
print("First 5 records:")
print(df.head())

First 5 records:
   ID   airline   flight source_city departure_time stops   arrival_time  \
0   0  SpiceJet  SG-8709       Delhi        Evening  zero          Night   
1   1  SpiceJet  SG-8157       Delhi  Early_Morning  zero        Morning   
2   2   AirAsia   I5-764       Delhi  Early_Morning  zero  Early_Morning   
3   3   Vistara   UK-995       Delhi        Morning  zero      Afternoon   
4   4   Vistara   UK-963       Delhi        Morning  zero        Morning   

  destination_city    class  duration  days_left  price  
0           Mumbai  Economy      2.17          1   5953  
1           Mumbai  Economy      2.33          1   5953  
2           Mumbai  Economy      2.17          1   5956  
3           Mumbai  Economy      2.25          1   5955  
4           Mumbai  Economy      2.33          1   5955  


In [5]:
print("DataFrame:")
print(df.info())

DataFrame:
<class 'pandas.DataFrame'>
RangeIndex: 300153 entries, 0 to 300152
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   ID                300153 non-null  int64  
 1   airline           300153 non-null  str    
 2   flight            300153 non-null  str    
 3   source_city       300153 non-null  str    
 4   departure_time    300153 non-null  str    
 5   stops             300153 non-null  str    
 6   arrival_time      300153 non-null  str    
 7   destination_city  300153 non-null  str    
 8   class             300153 non-null  str    
 9   duration          300153 non-null  float64
 10  days_left         300153 non-null  int64  
 11  price             300153 non-null  int64  
dtypes: float64(1), int64(3), str(8)
memory usage: 27.5 MB
None


In [6]:
print("Summary statistics:")
print(df.describe())

Summary statistics:
                  ID       duration      days_left          price
count  300153.000000  300153.000000  300153.000000  300153.000000
mean   150076.000000      12.221021      26.004751   20889.660523
std     86646.852011       7.191997      13.561004   22697.767366
min         0.000000       0.830000       1.000000    1105.000000
25%     75038.000000       6.830000      15.000000    4783.000000
50%    150076.000000      11.250000      26.000000    7425.000000
75%    225114.000000      16.170000      38.000000   42521.000000
max    300152.000000      49.830000      49.000000  123071.000000


In [7]:
print("Shape - rows, columns:")
print(df.shape)

Shape - rows, columns:
(300153, 12)


In [8]:
print("Column data types:")
print(df.dtypes)

Column data types:
ID                    int64
airline                 str
flight                  str
source_city             str
departure_time          str
stops                   str
arrival_time            str
destination_city        str
class                   str
duration            float64
days_left             int64
price                 int64
dtype: object


In [10]:
print("Number of unique entries per column:")
print(df.nunique())

Number of unique entries per column:
ID                  300153
airline                  6
flight                1561
source_city              6
departure_time           6
stops                    3
arrival_time             6
destination_city         6
class                    2
duration               476
days_left               49
price                12157
dtype: int64


In [ ]:
#step 2 Clean 
print("Step 2: Clean Data")

In [ ]:
print("Null values per column:")
print(df.isnull().sum())

In [98]:
print("Duplicate rows:")
print(df.duplicated().sum())

Duplicate rows:
0


Step 3: Analysis

In [11]:
df.groupby('duration')[['source_city', 'destination_city', 'airline', 'flight']].nunique().sort_values('source_city', ascending=False)

,source_city,destination_city,airline,flight
duration,,,,
27.25,6,6,3,25
27.33,6,6,2,25
26.17,6,6,3,30
26.25,6,6,3,35
26.33,6,6,3,32
...,...,...,...,...
44.50,1,1,1,1
45.83,1,1,1,1
47.08,1,1,1,1


In [12]:
df.groupby('duration')[['source_city', 'destination_city', 'airline', 'flight']] \
  .nunique() \
  .assign(total_variety=lambda x: x.sum(axis=1)) \
  .sort_values('total_variety', ascending=False)


,source_city,destination_city,airline,flight,total_variety
duration,,,,,
2.25,5,5,6,98,114
2.17,5,4,6,99,114
2.08,5,6,6,77,94
2.75,5,5,6,76,92
6.33,6,6,6,72,90
...,...,...,...,...,...
44.50,1,1,1,1,4
45.83,1,1,1,1,4
47.08,1,1,1,1,4


In [13]:
subset_cols = ['flight', 'airline', 'source_city', 'destination_city', 'departure_time']

# 1. Exact duplicate rows on flight + airline + route + time of day
dupe_count = df.duplicated(subset=subset_cols).sum()
print(f"Duplicate rows on {subset_cols}: {dupe_count}")

if dupe_count > 0:
    print(df[df.duplicated(subset=subset_cols, keep=False)].sort_values(subset_cols))

# 2. Check if the same flight number + airline is tied to more than one route or time of day
route_variety = df.groupby(['flight', 'airline'])[['source_city', 'destination_city', 'departure_time']].nunique()

route_variety['route_varies'] = (route_variety['source_city'] > 1) | (route_variety['destination_city'] > 1)
route_variety['time_varies'] = route_variety['departure_time'] > 1

print(f"\nFlight/airline pairs linked to more than one route or departure time: {(route_variety['route_varies'] | route_variety['time_varies']).sum()}")
print(route_variety[['route_varies', 'time_varies']].value_counts())
print()
print(route_variety[route_variety['route_varies'] | route_variety['time_varies']])


Duplicate rows on ['flight', 'airline', 'source_city', 'destination_city', 'departure_time']: 296030
            ID  airline  flight source_city departure_time stops arrival_time  \
167218  167218   Indigo  6E-102   Hyderabad        Evening   one        Night   
167444  167444   Indigo  6E-102   Hyderabad        Evening   one        Night   
167667  167667   Indigo  6E-102   Hyderabad        Evening   one        Night   
168011  168011   Indigo  6E-102   Hyderabad        Evening   one        Night   
168236  168236   Indigo  6E-102   Hyderabad        Evening   one        Night   
...        ...      ...     ...         ...            ...   ...          ...   
237917  237917  Vistara  UK-996      Mumbai        Evening   one      Evening   
237982  237982  Vistara  UK-996      Mumbai        Evening   one      Morning   
237985  237985  Vistara  UK-996      Mumbai        Evening   one      Evening   
237987  237987  Vistara  UK-996      Mumbai        Evening   one      Evening   
237996  

In [19]:
cols = ['flight', 'airline', 'source_city', 'destination_city', 'departure_time', 'arrival_time', 'duration', 'stops']

# Table of unique flights, defined by flight number + airline + route + schedule details
unique_flights = df[cols].drop_duplicates().reset_index(drop=True)

print("Number of unique flights:", len(unique_flights))
print(unique_flights.head(10))

# Attach how many rows in the full dataset correspond to each unique flight
counts = df.groupby(cols).size().reset_index(name='row_count')
unique_flights = unique_flights.merge(counts, on=cols)
print(unique_flights.sort_values('row_count', ascending=False).head(10))

print("\nShape (rows, columns):", unique_flights.shape)
print("\nTable info:")
print(unique_flights.info())

Number of unique flights: 12955
    flight   airline source_city destination_city departure_time  \
0  SG-8709  SpiceJet       Delhi           Mumbai        Evening   
1  SG-8157  SpiceJet       Delhi           Mumbai  Early_Morning   
2   I5-764   AirAsia       Delhi           Mumbai  Early_Morning   
3   UK-995   Vistara       Delhi           Mumbai        Morning   
4   UK-963   Vistara       Delhi           Mumbai        Morning   
5   UK-945   Vistara       Delhi           Mumbai        Morning   
6   UK-927   Vistara       Delhi           Mumbai        Morning   
7   UK-951   Vistara       Delhi           Mumbai      Afternoon   
8   G8-334  GO_FIRST       Delhi           Mumbai  Early_Morning   
9   G8-336  GO_FIRST       Delhi           Mumbai      Afternoon   

    arrival_time  duration stops  
0          Night      2.17  zero  
1        Morning      2.33  zero  
2  Early_Morning      2.17  zero  
3      Afternoon      2.25  zero  
4        Morning      2.33  zero  
5      Af

In [5]:
print("1. Most popular routes (by row count)")
print(df["route"].value_counts().head(10))



1. Most popular routes (by row count)
route
Delhi -> Mumbai        15289
Mumbai -> Delhi        14809
Delhi -> Bangalore     14012
Bangalore -> Delhi     13756
Bangalore -> Mumbai    12939
Mumbai -> Bangalore    12885
Mumbai -> Kolkata      12602
Delhi -> Kolkata       11934
Kolkata -> Mumbai      11467
Delhi -> Chennai       10780
Name: count, dtype: int64


In [22]:
print("2a. Business vs Economy price gap")
print(df.groupby("class")["price"].mean())

print()
print("Controlling for route (Delhi -> Mumbai) and normalizing per hour of duration:")
df["price_per_hour"] = df["price"] / df["duration"]
sub = df[df["route"] == "Delhi -> Mumbai"]
print(sub.groupby("class")[["price", "duration"]].mean())
print(df.groupby("class")["price_per_hour"].mean())

2a. Business vs Economy price gap
class
Business    52540.081124
Economy      6572.342383
Name: price, dtype: float64

Controlling for route (Delhi -> Mumbai) and normalizing per hour of duration:
                 price   duration
class                            
Business  44364.442811  11.930388
Economy    6059.826087   9.537000
class
Business    5166.682677
Economy      822.360767
Name: price_per_hour, dtype: float64


In [23]:
print("2b. Business vs Economy price gap")
print(df.groupby("class")["price"].mean())

print()
print("Normalizing per hour of duration:")
df["price_per_hour"] = df["price"] / df["duration"]
print(df.groupby("class")["price_per_hour"].mean())

2b. Business vs Economy price gap
class
Business    52540.081124
Economy      6572.342383
Name: price, dtype: float64

Normalizing per hour of duration:
class
Business    5166.682677
Economy      822.360767
Name: price_per_hour, dtype: float64


In [15]:
print("3. Stops vs price")
print(df.groupby("stops")["price"].mean())

3. Stops vs price
stops
one            22900.992482
two_or_more    14113.450775
zero            9375.938535
Name: price, dtype: float64


In [29]:
print("4a. Price vs booking window (days_left)")
bins = [0, 3, 7, 15, 30, 50]
df["days_left_bucket"] = pd.cut(df["days_left"], bins)
print(df.groupby("days_left_bucket", observed=True)["price"].mean())

4a. Price vs booking window (days_left)
days_left_bucket
(0, 3]      28068.681894
(3, 7]      25698.242059
(7, 15]     23534.110876
(15, 30]    19754.193214
(30, 50]    19260.919021
Name: price, dtype: float64


In [31]:
print("4b. Average price per day (days_left)")
print(df.groupby("days_left")["price"].mean())

4b. Average price per day (days_left)
days_left
1     21591.867151
2     30211.299801
3     28976.083569
4     25730.905653
5     26679.773368
6     24856.493902
7     25588.367351
8     24895.883995
9     25726.246072
10    25572.819134
11    22990.656070
12    22505.803322
13    22498.885384
14    22678.002363
15    21952.540852
16    20503.546237
17    20386.353949
18    19987.445168
19    19507.677375
20    19699.983390
21    19430.494058
22    19590.667385
23    19840.913451
24    19803.908896
25    19571.641791
26    19238.290278
27    19950.866195
28    19534.986047
29    19744.653119
30    19567.580834
31    19392.706612
32    19258.135308
33    19306.271739
34    19562.008266
35    19255.652996
36    19517.688444
37    19506.306516
38    19734.912316
39    19262.095556
40    19144.972439
41    19347.440460
42    19154.261659
43    19340.528894
44    19049.080174
45    19199.876307
46    19305.351623
47    18553.272038
48    18998.126851
49    18992.971888
Name: price, dtype: f

In [32]:
print("4c. Average price and number of flights per day (days_left)")
print(df.groupby("days_left")["price"].agg(["mean", "count"]).rename(columns={"mean": "avg_price", "count": "num_flights"}))

4c. Average price and number of flights per day (days_left)
              avg_price  num_flights
days_left                           
1          21591.867151         1927
2          30211.299801         4026
3          28976.083569         4248
4          25730.905653         5077
5          26679.773368         5392
6          24856.493902         5740
7          25588.367351         5703
8          24895.883995         5767
9          25726.246072         5665
10         25572.819134         5822
11         22990.656070         6417
12         22505.803322         6381
13         22498.885384         6404
14         22678.002363         6349
15         21952.540852         6340
16         20503.546237         6272
17         20386.353949         6419
18         19987.445168         6602
19         19507.677375         6537
20         19699.983390         6502
21         19430.494058         6479
22         19590.667385         6494
23         19840.913451         6401
24         1980

In [33]:

print ("4d. Class mix and average price per class, day 1 vs day 2")
d1 = df[df["days_left"] == 1]
d2 = df[df["days_left"] == 2]

print("Class mix, day 1 vs day 2:")
print(pd.concat([
    d1["class"].value_counts(normalize=True).rename("day1"),
    d2["class"].value_counts(normalize=True).rename("day2"),
], axis=1))

print("\nAverage price WITHIN each class:")
print(pd.concat([
    d1.groupby("class")["price"].mean().rename("day1"),
    d2.groupby("class")["price"].mean().rename("day2"),
], axis=1))


4d. Class mix and average price per class, day 1 vs day 2
Class mix, day 1 vs day 2:
              day1     day2
class                      
Economy   0.861962  0.65077
Business  0.138038  0.34923

Average price WITHIN each class:
                 day1          day2
class                              
Business  65169.31203  60455.848506
Economy   14613.17941  13980.828244


In [42]:
print("5. Cheapest and priciest routes on average (min 500 rows)")
route_stats = df.groupby("route")["price"].agg(["mean", "count"]).rename(columns={"mean": "Avg Price", "count": "Flight Count"})
route_stats = route_stats[route_stats["Flight Count"] > 500]
print("Cheapest:")
print(route_stats.sort_values("Avg Price").head(3))
print()
print("Priciest:")
print(route_stats.sort_values("Avg Price", ascending=False).head(3))

5. Cheapest and priciest routes on average (min 500 rows)
Cheapest:
                       Avg Price  Flight Count
route                                         
Hyderabad -> Delhi  17243.945685          8506
Delhi -> Hyderabad  17347.288379          9328
Bangalore -> Delhi  17723.313972         13756

Priciest:
                         Avg Price  Flight Count
route                                           
Chennai -> Bangalore  25081.850454          6493
Kolkata -> Chennai    23660.361040          6653
Bangalore -> Kolkata  23500.061229         10028


In [56]:
print("6. Airline market share on the Delhi -> Mumbai route")
result = pd.crosstab(df["route"], df["airline"], normalize="index").loc["Delhi -> Mumbai"]*100
result = result.to_frame("Share%")
print(result)

6. Airline market share on the Delhi -> Mumbai route
              Share%
airline             
AirAsia     4.133691
Air_India  32.749035
GO_FIRST   10.792073
Indigo     10.831317
SpiceJet    3.296488
Vistara    38.197397


In [58]:
print("6b. Airline market share on the Delhi -> Mumbai route")

flight_cols = ["flight", "airline", "source_city", "destination_city",
               "departure_time", "arrival_time", "duration", "stops"]
unique_flights = df[flight_cols].drop_duplicates().copy()
unique_flights["route"] = unique_flights["source_city"] + " -> " + unique_flights["destination_city"]

result = pd.crosstab(unique_flights["route"], unique_flights["airline"], normalize="index").loc["Delhi -> Mumbai"]*100
result = result.to_frame("Share%")
print(result)


6b. Airline market share on the Delhi -> Mumbai route
              Share%
airline             
AirAsia     4.537205
Air_India  33.938294
GO_FIRST   18.874773
Indigo     15.245009
SpiceJet    4.355717
Vistara    23.049002


In [65]:
print("6c. Airline market share Overall per routes")

print(pd.crosstab(df["route"], df["airline"], normalize="index"))


6c. Airline market share Overall per routes
airline                  AirAsia  Air_India  GO_FIRST    Indigo  SpiceJet  \
route                                                                       
Bangalore -> Chennai    0.021529   0.251170  0.057566  0.043994  0.009048   
Bangalore -> Delhi      0.113260   0.191407  0.105118  0.142556  0.053431   
Bangalore -> Hyderabad  0.021505   0.249552  0.052979  0.147625  0.003472   
Bangalore -> Kolkata    0.084563   0.202034  0.087056  0.163243  0.027323   
Bangalore -> Mumbai     0.048535   0.274751  0.103331  0.145452  0.012134   
Chennai -> Bangalore    0.021254   0.252272  0.058833  0.053596  0.008471   
Chennai -> Delhi        0.072268   0.188592  0.058878  0.178575  0.077992   
Chennai -> Hyderabad    0.020646   0.234639  0.006226  0.230051  0.008357   
Chennai -> Kolkata      0.043964   0.267650  0.024631  0.220679  0.020765   
Chennai -> Mumbai       0.023560   0.334975  0.012958  0.182694  0.021953   
Delhi -> Bangalore      0.110548

In [68]:
print("6d. Airline market share Overall")

overall_share = df["airline"].value_counts(normalize=True).mul(100).round(1)
print(overall_share)

6d. Airline market share Overall
airline
Vistara      42.6
Air_India    27.0
Indigo       14.4
GO_FIRST      7.7
AirAsia       5.4
SpiceJet      3.0
Name: proportion, dtype: float64


In [89]:
print("7a. Duration vs price correlation, controlling for stops")
print(df.groupby("stops")[["duration", "price"]].corr().iloc[0::2, -1])

7a. Duration vs price correlation, controlling for stops
stops                
one          duration    0.140521
two_or_more  duration    0.010959
zero         duration    0.222263
Name: price, dtype: float64


In [90]:
print("7b. Duration vs price, controlling for stops")
bins = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
df["duration_bucket"] = pd.cut(df["duration"], bins)

print("\n")

print(df.groupby("duration_bucket", observed=True)["price"].agg(["mean", "count"]))

7b. Duration vs price, controlling for stops


                         mean  count
duration_bucket                     
(0, 5]            9524.700547  45513
(5, 10]          17771.231946  84761
(10, 15]         24933.603765  80220
(15, 20]         27373.403712  38904
(20, 25]         26526.462091  29808
(25, 30]         23284.224507  18721
(30, 35]         17986.455262   1777
(35, 40]         14708.960100    401
(40, 45]         12920.850000     40
(45, 50]         24848.625000      8


In [92]:
print("7c. Number of flights per duration bucket and class")
print(pd.crosstab(df["duration_bucket"], df["class"]))

7c. Number of flights per duration bucket and class
class            Business  Economy
duration_bucket                   
(0, 5]               9333    36180
(5, 10]             19308    65453
(10, 15]            29224    50996
(15, 20]            16339    22565
(20, 25]            12383    17425
(25, 30]             6620    12101
(30, 35]              250     1527
(35, 40]               27      374
(40, 45]                1       39
(45, 50]                2        6


In [96]:
print("7d. Mean price per duration bucket and class\n")
result = pd.crosstab(df["duration_bucket"], df["class"], values=df["price"], aggfunc="mean").round(2)
result["sum"] = result["Business"]+result["Economy"]
print(result)

7d. Mean price per duration bucket and class

class            Business   Economy       sum
duration_bucket                              
(0, 5]           29511.76   4368.83  33880.59
(5, 10]          56069.77   6473.54  62543.31
(10, 15]         56271.13   6975.18  63246.31
(15, 20]         54914.68   7431.15  62345.83
(20, 25]         53017.61   7700.65  60718.26
(25, 30]         51180.41   8023.27  59203.68
(30, 35]         58874.68  11292.25  70166.93
(35, 40]         68340.93  10837.13  79178.06
(40, 45]         65674.00  11568.21  77242.21
(45, 50]         53525.00  15289.83  68814.83


In [14]:
print("1. Most popular routes (by distinct flights, not raw rows)")

flight_cols = [
    "flight", "airline", "source_city", "destination_city",
    "departure_time", "arrival_time", "duration", "stops"
]

unique_flights = df[flight_cols].drop_duplicates().copy()
unique_flights["route"] = (
    unique_flights["source_city"] + " -> " +
    unique_flights["destination_city"]
)

print(unique_flights["route"].value_counts().head(10))


1. Most popular routes (by distinct flights, not raw rows)
route
Delhi -> Bangalore     640
Mumbai -> Bangalore    636
Bangalore -> Delhi     618
Mumbai -> Kolkata      579
Bangalore -> Mumbai    573
Kolkata -> Mumbai      553
Delhi -> Mumbai        551
Mumbai -> Delhi        541
Delhi -> Kolkata       529
Kolkata -> Delhi       510
Name: count, dtype: int64
route
Delhi -> Bangalore     640
Mumbai -> Bangalore    636
Bangalore -> Delhi     618
Mumbai -> Kolkata      579
Bangalore -> Mumbai    573
Kolkata -> Mumbai      553
Delhi -> Mumbai        551
Mumbai -> Delhi        541
Delhi -> Kolkata       529
Kolkata -> Delhi       510
Name: count, dtype: int64
